In [18]:
# 读取环境变量
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
# 定义模型
import os
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key
)

In [20]:
# 定义工具
from langchain_tavily import TavilySearch
search_tool = TavilySearch(
    max_results=5,
    topic="general"
)

In [21]:
# 系统提示词
system_prompt = """
你是一名专业的私厨管家，请严格按照以下流程执行：
1. 食材识别与甄选：列出用户当前可用的食材清单，如果用户有上传图片，分析图片中的内容，根据食材外观和新鲜程度，分析出可用的食材清单。
2. 联网搜索食谱：使用web_search工具，查询关键词为step1得出的可用食材清单，搜素相关联的食谱。
3. 智能排序与筛选：根据食谱，从制作难度和营养成分两个角度综合量化打分，根据得分从高到低给出TOP3食谱
4. 生成食谱分析报告：要求包含食谱名称、食材清单、制作难度、营养成分、图片样本参考、制作步骤
"""

In [25]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 定义记忆
from langgraph.checkpoint.sqlite import SqliteSaver
with SqliteSaver.from_conn_string("personal_chef_checkpoints.db") as check_pointer:
    check_pointer.setup()
    agent = create_agent(
        model=model,
        tools=[search_tool],
        checkpointer=check_pointer,
        system_prompt=system_prompt
    )
    resp = agent.invoke({
        "messages": [
            HumanMessage([
                {"type": "text", "text": "帮我看看能做什么？"},
                {"type": "image", "url": "https://assets.699pic.com/public/img95/photo/60024/7565.jpg_wh860.jpg"},
            ])
        ]
    },{"configurable": {"thread_id": "1"}})
    for message in resp["messages"]:
        message.pretty_print()


================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image', 'url': 'https://assets.699pic.com/public/img95/photo/60024/7565.jpg_wh860.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image', 'url': 'https://assets.699pic.com/public/img95/photo/60024/7565.jpg_wh860.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image', 'url': 'https://assets.699pic.com/public/img95/photo/60024/7565.jpg_wh860.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image', 'url': 'https://assets.699pic.com/public/img95/photo/60024/7565.jpg_wh860.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_6

## Langsmith 接入 agent 调试部署
### 接入步骤
1. 准备 langsmith 对应的 api_key 并配置环境变量
    ```
    LANGSMITH_API_KEY=xxx
    LANGSMITH_TACEING=true
    LANGSMITH_PROJECT=personal-chef
    ```
2. 编写 agent 对应代码 py 文件
* 注意无需定义 checkpoint ，因为 langgraph 会自动引入记忆
* 注意只需要定义 agent，无需手动调用
3. 安装 langgraph 本地部署工具：uv add langgraph-cli[inmem]
4. 在项目目录创建 langgraph 配置文件，名称为: langgraph.json，内容如下：
    ```
    {
        "dependencies": ["."],
        "graphs": {
            "chef_agent": "./personal_chef.py:agent"
        },
        "env": ".env"
    }
    ```
5. 启动运行：uv run langgraph dev



## 接入 FastAPI 和 OSS
1. 安装依赖：uv add fastapi alibabacloud-oss-v2